CONFIGURATION

In [0]:

# SUPPRESS WARNINGS

import warnings
import os

# Suppress Python deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Suppress threadpoolctl errors (Databricks-specific issue)
os.environ["OMP_NUM_THREADS"] = "1"

# Suppress MLflow signature warnings
warnings.filterwarnings("ignore", message=".*Inferred schema contains integer.*")
warnings.filterwarnings("ignore", message=".*Model logged without a signature.*")

# Suppress ipykernel deprecation
warnings.filterwarnings("ignore", message=".*ipykernel.comm.Comm.*")

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"


print("✅ Warnings suppressed")

✅ Warnings suppressed


In [0]:
# Config
STORAGE_ACCOUNT = "dnvenergystorage"
CONTAINER = "building-data"
RESULTS = "results"
ACCOUNT_KEY = "uR0lF4HmJo/BqqH/EDo5qzgzkoGduiiGHTL94B3ZrTWvsx/l20Us9oaX/4BxtKPCeXVF4OMkswna+AStxmynZQ==" 

BASE_PATH = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"
SAVE_PATH = f"wasbs://{RESULTS}@{STORAGE_ACCOUNT}.blob.core.windows.net"

START_DATE = "2017-01-01"
END_DATE = "2017-07-01"

# Connect to Azure
spark._jsc.hadoopConfiguration().set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net",
    ACCOUNT_KEY
)

from pyspark.sql import functions as F
from pyspark.sql.window import Window



1. DATA INGESTION


 Uploaded Data From Kaggle To Azure Blob. If real batch data use Airlfow or Azure Data Factory. If streaming use Kafka or Azure Event Hubs. ADF is an orchestrator that moves data between sources, schedules workflows and reacts to events (like EventBridge or EventGrid or cron) and manage dependencies.

If data gets more complex than just appending, one could use CDC.


We have 1500 buildings with hourly observations, around 7M rows. 

- Electricity (Meter)
- Cooling (HVAC)
- Heating (HVAC)
- Weather (Air Temperature)

2. RAW DATA STORAGE

Use Blob as storage for raw data. Electricity, Weather, Heating, Cooling data. This is the Bronze Layer. 

In a more complex project use ADLS with Delta Lake 

ADLS adds analytics on top of the storage and some hierarchy
Azure Data Factory and Delta Lake. Blob is just a bucket. 

Delta only makes sense if you are receiving streaming data or merging different sources. It would add an additional source of truth on top of the ingested data. In the current case, the ingested data is static and single source of truth.


In [0]:
# Config - set to None for all buildings, or a number for fast testing
N_BUILDINGS = None  # Set to None to use all buildings

# Load raw data
df_elec = spark.read.csv(f"{BASE_PATH}/electricity_cleaned.csv", header=True, inferSchema=True)
df_weather = spark.read.csv(f"{BASE_PATH}/weather.csv", header=True, inferSchema=True)

print(f"✅ Electricity: {df_elec.count():,} rows")
print(f"✅ Weather: {df_weather.count():,} rows")

# Get numeric building columns
numeric_cols = [c for c in df_elec.columns if c != 'timestamp' 
                and dict(df_elec.dtypes)[c] in ['double', 'float', 'int', 'bigint']]

# Sample buildings if N_BUILDINGS is set
if N_BUILDINGS is not None:
    numeric_cols = numeric_cols[:N_BUILDINGS]
    print(f"✅ Buildings: {len(numeric_cols)} (sampled from full dataset)")
else:
    print(f"✅ Buildings: {len(numeric_cols)}")

# Filter date range and unpivot to long format
df_elec_filtered = df_elec.select(['timestamp'] + numeric_cols).filter(
    (F.col('timestamp') >= START_DATE) & (F.col('timestamp') < END_DATE)
)

stack_expr = "stack({}, {}) as (building_id, meter_reading)".format(
    len(numeric_cols),
    ', '.join([f"'{b}', `{b}`" for b in numeric_cols])
)

df_long = df_elec_filtered.selectExpr('timestamp', stack_expr).filter(
    F.col('meter_reading').isNotNull()
)

print(f"✅ Bronze layer: {df_long.count():,} readings")

# Load HVAC data (optional)
df_cooling, df_heating = None, None

try:
    df_cooling_raw = spark.read.csv(f"{BASE_PATH}/chilledwater_cleaned.csv", header=True, inferSchema=True)
    cooling_cols = [c for c in df_cooling_raw.columns if c != 'timestamp' 
                   and dict(df_cooling_raw.dtypes).get(c) in ['double', 'float', 'int', 'bigint']
                   and c in numeric_cols]  # Already filtered by N_BUILDINGS
    if cooling_cols:
        df_cooling_filtered = df_cooling_raw.select(['timestamp'] + cooling_cols).filter(
            (F.col('timestamp') >= START_DATE) & (F.col('timestamp') < END_DATE))
        stack_cool = "stack({}, {}) as (building_id, cooling_reading)".format(
            len(cooling_cols), ', '.join([f"'{b}', `{b}`" for b in cooling_cols]))
        df_cooling = df_cooling_filtered.selectExpr('timestamp', stack_cool).filter(F.col('cooling_reading').isNotNull())
        print(f"✅ Cooling: {df_cooling.select('building_id').distinct().count()} buildings")
except:
    print("⚠️ No cooling data")

try:
    df_heating_raw = spark.read.csv(f"{BASE_PATH}/hotwater_cleaned.csv", header=True, inferSchema=True)
    heating_cols = [c for c in df_heating_raw.columns if c != 'timestamp'
                   and dict(df_heating_raw.dtypes).get(c) in ['double', 'float', 'int', 'bigint']
                   and c in numeric_cols]  # Already filtered by N_BUILDINGS
    if heating_cols:
        df_heating_filtered = df_heating_raw.select(['timestamp'] + heating_cols).filter(
            (F.col('timestamp') >= START_DATE) & (F.col('timestamp') < END_DATE))
        stack_heat = "stack({}, {}) as (building_id, heating_reading)".format(
            len(heating_cols), ', '.join([f"'{b}', `{b}`" for b in heating_cols]))
        df_heating = df_heating_filtered.selectExpr('timestamp', stack_heat).filter(F.col('heating_reading').isNotNull())
        print(f"✅ Heating: {df_heating.select('building_id').distinct().count()} buildings")
except:
    print("⚠️ No heating data")

# =============================================================================
# BRONZE DELTA LAKE STORAGE
# =============================================================================
# In a production pipeline with proper data ingestion (streaming or scheduled 
# batch loads), you would persist Bronze layer to Delta Lake here.
#
# Benefits:
# - ACID transactions: Safe concurrent writes from multiple sources
# - Time travel: Query data as it existed at any point in time
# - Schema enforcement: Reject malformed data before it enters pipeline
# - Audit trail: Track when each record was ingested
#
# For this POC using static CSVs, the source files already serve as 
# source of truth. In production with real-time meter data:
# =============================================================================

# df_long.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/delta/bronze/meter_readings")
# print(f"✅ Bronze meter readings saved to Delta")

# if df_cooling is not None:
#     df_cooling.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/delta/bronze/cooling_readings")
#     print(f"✅ Bronze cooling readings saved to Delta")

# if df_heating is not None:
#     df_heating.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/delta/bronze/heating_readings")
#     print(f"✅ Bronze heating readings saved to Delta")

✅ Electricity: 17,544 rows
✅ Weather: 331,166 rows
✅ Buildings: 1572
✅ Bronze layer: 6,327,174 readings
✅ Cooling: 526 buildings
✅ Heating: 180 buildings


3. ETL 

This is the Silver Layer. Use PySpark to clean and join the datasets from previous step. Then load back the transformed data into structured tables using DeltaLake, ready for anlaytics layer.


One could keep working with the envirnoment variable, but DeltaLake adds this additional layer of truth, on top of allowing you to store the data without having to rerun everything.


Here, the hourly data is transformed into Daily Data per building, using key statistics such as mean, max, min or rolling averages.

In [0]:
print("=" * 60)
print("STAGE 3: ETL - SILVER LAYER")
print("=" * 60)

# Temporal features
df_long = df_long \
    .withColumn('hour', F.hour('timestamp')) \
    .withColumn('day_of_week', F.dayofweek('timestamp')) \
    .withColumn('is_weekend', F.when(F.col('day_of_week').isin([1, 7]), 1).otherwise(0)) \
    .withColumn('is_night', F.when((F.col('hour') >= 23) | (F.col('hour') <= 5), 1).otherwise(0)) \
    .withColumn('is_business_hours', F.when((F.col('hour') >= 8) & (F.col('hour') <= 18) & 
                                            (~F.col('day_of_week').isin([1, 7])), 1).otherwise(0)) \
    .withColumn('site', F.split(F.col('building_id'), '_').getItem(0)) \
    .withColumn('building_type', F.split(F.col('building_id'), '_').getItem(1))

# Weather join (broadcast for efficiency)
df_weather_clean = df_weather.select(
    F.col('timestamp').alias('weather_ts'),
    F.col('site_id').alias('weather_site'),
    F.col('airTemperature').alias('air_temperature')
).dropna()

df_long = df_long.join(
    F.broadcast(df_weather_clean),
    (df_long['timestamp'] == df_weather_clean['weather_ts']) & 
    (df_long['site'] == df_weather_clean['weather_site']),
    'left'
).drop('weather_ts', 'weather_site')

# Rolling features
window_7d = Window.partitionBy('building_id').orderBy('timestamp').rowsBetween(-168, 0)
window_24h = Window.partitionBy('building_id').orderBy('timestamp').rowsBetween(-24, 0)

df_long = df_long \
    .withColumn('baseload_7day', F.expr('percentile_approx(meter_reading, 0.1)').over(window_7d)) \
    .withColumn('rolling_avg_24h', F.avg('meter_reading').over(window_24h)) \
    .withColumn('rolling_std_24h', F.stddev('meter_reading').over(window_24h)) \
    .withColumn('peak_ratio', F.col('meter_reading') / (F.col('baseload_7day') + 0.01)) \
    .withColumn('volatility', F.col('rolling_std_24h') / (F.col('rolling_avg_24h') + 0.01))

# Daily aggregation
df_daily = df_long.groupBy('building_id', F.to_date('timestamp').alias('date')).agg(
    F.mean('meter_reading').alias('avg_consumption'),
    F.max('meter_reading').alias('peak_consumption'),
    F.min('meter_reading').alias('min_consumption'),
    F.stddev('meter_reading').alias('consumption_std'),
    F.mean('baseload_7day').alias('baseload'),
    F.mean('peak_ratio').alias('avg_peak_ratio'),
    F.mean('volatility').alias('avg_volatility'),
    F.mean('air_temperature').alias('avg_temp'),
    F.first('building_type').alias('building_type'),
    F.max('is_weekend').alias('is_weekend')
)

# Multi-meter integration
if df_cooling is not None:
    df_cooling_daily = df_cooling.groupBy('building_id', F.to_date('timestamp').alias('date')).agg(
        F.mean('cooling_reading').alias('cooling_avg'))
    df_daily = df_daily.join(df_cooling_daily, ['building_id', 'date'], 'left')

if df_heating is not None:
    df_heating_daily = df_heating.groupBy('building_id', F.to_date('timestamp').alias('date')).agg(
        F.mean('heating_reading').alias('heating_avg'))
    df_daily = df_daily.join(df_heating_daily, ['building_id', 'date'], 'left')

df_daily.cache()
print(f"✅ Silver layer: {df_daily.count():,} daily records, {df_daily.select('building_id').distinct().count()} buildings")

# Write Silver to Delta Lake
# Why here: Silver is cleaned, transformed data - worth persisting for:
# - Reprocessing Gold layer without re-running ETL
# - MERGE for late-arriving meter corrections
# - Time travel to compare model performance across data versions
df_daily.write.format("delta").mode("overwrite").save(f"{BASE_PATH}/delta/silver/daily_consumption")
print(f"✅ Silver saved to Delta: {BASE_PATH}/delta/silver/daily_consumption")


STAGE 3: ETL - SILVER LAYER
✅ Silver layer: 265,294 daily records, 1534 buildings
✅ Silver saved to Delta: wasbs://building-data@dnvenergystorage.blob.core.windows.net/delta/silver/daily_consumption


4. DATA VALIDATION


Use Great Expectations to validate data before proceeding: minimum number of buidlings, 95% nans at most, existance of id columns and no negative values. 

In [0]:
#%pip install great-expectations
import great_expectations as gx

df_daily_pd = df_daily.toPandas()

# Ephemeral (in-memory) context
context = gx.get_context(mode="ephemeral")

# Pandas datasource
datasource = context.data_sources.add_pandas("pandas_ds")

# DataFrame asset
asset = datasource.add_dataframe_asset(name="daily_data")

# Batch
batch = (
    asset.add_batch_definition_whole_dataframe("daily_batch")
    .get_batch({"dataframe": df_daily_pd})
)

# Create expectation suite explicitly
context.suites.add(gx.ExpectationSuite(name="daily_expectations"))

# Validator
validator = context.get_validator(
    batch=batch,
    expectation_suite_name="daily_expectations",
)

# Expectations
results = [
    validator.expect_table_row_count_to_be_between(min_value=1000),
    validator.expect_column_values_to_not_be_null("avg_consumption", mostly=0.95),
    validator.expect_column_values_to_be_between("avg_consumption", min_value=0),
    validator.expect_column_to_exist("building_id"),
]

for r in results:
    print(("✅" if r.success else "❌"), r.expectation_config.type)

assert all(r.success for r in results), "Validation FAILED"
print("\n✅ Validation PASSED")


/databricks/spark/python/pyspark/sql/pandas/utils.py:43: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):
/databricks/spark/python/pyspark/sql/pandas/utils.py:77: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pyarrow.__version__) < LooseVersion(minimum_pyarrow_version):
/databricks/python_shell/lib/dbruntime/safe_oinspect.py:133: DeprecationWarning: `getargspec` function is deprecated as of IPython 7.10and will be removed in future versions.
  argspec = oinspect.getargspec(obj)
/databricks/spark/python/pyspark/sql/pandas/conversion.py:161: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pa.__version__) >= LooseVersion("13.0.0"):
INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpcasprbd0' for ephemeral docs

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

✅ expect_table_row_count_to_be_between
✅ expect_column_values_to_not_be_null
✅ expect_column_values_to_be_between
✅ expect_column_to_exist

✅ Validation PASSED


5. FEATURE ENGINEERING


This is the Gold Layer. Use PySpark to create problem related features. In ETL you do standard cleaning and processing steps that always apply. Here you add features and cleaning in order to feed the data to your ML algorithm.

- Averages
- Ratios
- Standarization
- Normalization
- Train/Validation/Test Splits
- Etc

In this case I normalize and split later because I am running different models. 

We compute descriptive statistics per building using their daily features, so we end up with around 28 clean features and 1500 observations.

The clean dataset is finally stored in DeltaLake. However for more complex projects (real-time or multiple teams) we could consider a Feature Store shuc as Databricks FS. In this case, for batch training Delta Lake adds the persistence and versioning we need.

In [0]:
print("=" * 60)
print("STAGE 5: FEATURE ENGINEERING - GOLD LAYER")
print("=" * 60)

# Building-level aggregations
df_gold = df_daily.groupBy('building_id').agg(
    F.mean('avg_consumption').alias('avg_consumption'),
    F.stddev('avg_consumption').alias('consumption_std'),
    F.mean('baseload').alias('baseload'),
    F.mean('peak_consumption').alias('peak_consumption'),
    F.mean('avg_peak_ratio').alias('avg_peak_ratio'),
    F.mean('avg_volatility').alias('avg_volatility'),
    F.mean('avg_temp').alias('avg_temp'),
    F.first('building_type').alias('building_type')
)

# Behavioral ratios
df_weekday = df_daily.filter(F.col('is_weekend') == 0).groupBy('building_id').agg(
    F.mean('avg_consumption').alias('weekday_avg'))
df_weekend = df_daily.filter(F.col('is_weekend') == 1).groupBy('building_id').agg(
    F.mean('avg_consumption').alias('weekend_avg'))
df_night = df_long.filter(F.col('is_night') == 1).groupBy('building_id').agg(
    F.mean('meter_reading').alias('night_avg'))
df_day = df_long.filter(F.col('is_night') == 0).groupBy('building_id').agg(
    F.mean('meter_reading').alias('day_avg'))
df_temp_sens = df_daily.groupBy('building_id').agg(
    F.corr('avg_temp', 'avg_consumption').alias('temp_sensitivity'))
df_cv = df_daily.groupBy('building_id').agg(
    (F.stddev('avg_consumption') / F.mean('avg_consumption')).alias('consumption_cv'))

# Join all features
for df_feat in [df_weekday, df_weekend, df_night, df_day, df_temp_sens, df_cv]:
    df_gold = df_gold.join(df_feat, 'building_id', 'left')

# Add HVAC features if available
if df_cooling is not None:
    df_cool_avg = df_daily.groupBy('building_id').agg(F.mean('cooling_avg').alias('cooling_avg'))
    df_gold = df_gold.join(df_cool_avg, 'building_id', 'left')
if df_heating is not None:
    df_heat_avg = df_daily.groupBy('building_id').agg(F.mean('heating_avg').alias('heating_avg'))
    df_gold = df_gold.join(df_heat_avg, 'building_id', 'left')

# Compute ratios
df_gold = df_gold \
    .withColumn('weekend_ratio', F.col('weekend_avg') / (F.col('weekday_avg') + 0.01)) \
    .withColumn('night_ratio', F.col('night_avg') / (F.col('day_avg') + 0.01))

# Convert to pandas
df_bldg = df_gold.toPandas()
print(f"✅ Gold layer: {len(df_bldg)} buildings, {len(df_bldg.columns)} features")


STAGE 5: FEATURE ENGINEERING - GOLD LAYER


/databricks/spark/python/pyspark/sql/pandas/utils.py:43: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):
/databricks/spark/python/pyspark/sql/pandas/utils.py:77: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pyarrow.__version__) < LooseVersion(minimum_pyarrow_version):
/databricks/python_shell/lib/dbruntime/safe_oinspect.py:133: DeprecationWarning: `getargspec` function is deprecated as of IPython 7.10and will be removed in future versions.
  argspec = oinspect.getargspec(obj)


✅ Gold layer: 1534 buildings, 19 features


/databricks/spark/python/pyspark/sql/pandas/conversion.py:161: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pa.__version__) >= LooseVersion("13.0.0"):


6. Business Analytics (Optional): The data is ready for ML. This is what Business Analysts would use to generate reports, etc.__ 

Normally, EDA and BI can be done before and/or after feature engineering. Depending on how you approach your data.

7. EDA


Use Pandas and Matplot to get a basic intuition of our data.
- Consumption Distribution
- Weekend vs Weekday Consumption
- Consumption by Building Type
- Baseload vs Peak Consumption
- Correlation Heatmap


In [0]:
print("=" * 60)
print("STAGE 7: EDA & VISUALIZATIONS")
print("=" * 60)

import matplotlib.pyplot as plt
import seaborn as sns

# Convert to pandas for plotting
df_daily_pd = df_daily.toPandas()
df_bldg_pd = df_bldg.copy()  # Already pandas from Gold layer

# Create output folder
import os
PLOTS_PATH = "/tmp/plots"
os.makedirs(PLOTS_PATH, exist_ok=True)

# -----------------------------------------------------------------------------
# 1. Consumption Distribution
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
df_bldg_pd['avg_consumption'].hist(bins=50, ax=ax, edgecolor='black')
ax.set_xlabel('Average Consumption (kWh)')
ax.set_ylabel('Number of Buildings')
ax.set_title('Distribution of Building Consumption')
plt.tight_layout()
plt.savefig(f"{PLOTS_PATH}/consumption_distribution.png", dpi=100)
plt.close()
print("✅ Saved: consumption_distribution.png")

# -----------------------------------------------------------------------------
# 2. Weekend vs Weekday Consumption
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 4))
weekend_data = df_daily_pd.groupby('is_weekend')['avg_consumption'].mean()
weekend_data.index = ['Weekday', 'Weekend']
weekend_data.plot(kind='bar', ax=ax, color=['steelblue', 'coral'], edgecolor='black')
ax.set_ylabel('Avg Consumption (kWh)')
ax.set_title('Weekday vs Weekend Consumption')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(f"{PLOTS_PATH}/weekend_vs_weekday.png", dpi=100)
plt.close()
print("✅ Saved: weekend_vs_weekday.png")

# -----------------------------------------------------------------------------
# 3. Consumption by Building Type
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 4))
type_consumption = df_bldg_pd.groupby('building_type')['avg_consumption'].mean().sort_values(ascending=False).head(10)
type_consumption.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_ylabel('Avg Consumption (kWh)')
ax.set_title('Top 10 Building Types by Consumption')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f"{PLOTS_PATH}/consumption_by_type.png", dpi=100)
plt.close()
print("✅ Saved: consumption_by_type.png")

# -----------------------------------------------------------------------------
# 4. Baseload vs Peak Ratio (Scatter)
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(df_bldg_pd['baseload'], df_bldg_pd['avg_peak_ratio'], alpha=0.5, s=20)
ax.set_xlabel('Baseload (kWh)')
ax.set_ylabel('Peak Ratio')
ax.set_title('Baseload vs Peak Ratio')
plt.tight_layout()
plt.savefig(f"{PLOTS_PATH}/baseload_vs_peak.png", dpi=100)
plt.close()
print("✅ Saved: baseload_vs_peak.png")

# -----------------------------------------------------------------------------
# 5. Correlation Heatmap
# -----------------------------------------------------------------------------
corr_cols = ['avg_consumption', 'baseload', 'weekend_ratio', 'night_ratio', 
             'consumption_cv', 'avg_volatility', 'temp_sensitivity']
corr_cols = [c for c in corr_cols if c in df_bldg_pd.columns]

fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = df_bldg_pd[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlations')
plt.tight_layout()
plt.savefig(f"{PLOTS_PATH}/correlation_heatmap.png", dpi=100)
plt.close()
print("✅ Saved: correlation_heatmap.png")

# -----------------------------------------------------------------------------
# 6. Summary Stats JSON (for API)
# -----------------------------------------------------------------------------
import json

summary_stats = {
    "total_buildings": len(df_bldg_pd),
    "avg_consumption_mean": round(df_bldg_pd['avg_consumption'].mean(), 2),
    "avg_consumption_std": round(df_bldg_pd['avg_consumption'].std(), 2),
    "baseload_mean": round(df_bldg_pd['baseload'].mean(), 2),
    "weekend_ratio_mean": round(df_bldg_pd['weekend_ratio'].mean(), 2) if 'weekend_ratio' in df_bldg_pd.columns else None,
    "building_types": df_bldg_pd['building_type'].nunique(),
    "top_building_types": df_bldg_pd['building_type'].value_counts().head(5).to_dict()
}

with open(f"{PLOTS_PATH}/summary_stats.json", 'w') as f:
    json.dump(summary_stats, f, indent=2)
print("✅ Saved: summary_stats.json")

# -----------------------------------------------------------------------------
# Copy to Azure (same as predictions)
# -----------------------------------------------------------------------------
for filename in os.listdir(PLOTS_PATH):
    dbutils.fs.cp(f"file:{PLOTS_PATH}/{filename}", f"{SAVE_PATH}/plots/{filename}")

print(f"\n✅ All plots saved to {SAVE_PATH}/plots/")

STAGE 7: EDA & VISUALIZATIONS


<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead
/databricks/spark/python/pyspark/sql/pandas/utils.py:43: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pandas.__version__) < LooseVersion(minimum_pandas_version):
/databricks/spark/python/pyspark/sql/pandas/utils.py:77: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pyarrow.__version__) < LooseVersion(minimum_pyarrow_version):
/databricks/spark/python/pyspark/sql/pandas/conversion.py:161: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(pa.__version__) >= LooseVersion("13.0.0"):


✅ Saved: consumption_distribution.png
✅ Saved: weekend_vs_weekday.png
✅ Saved: consumption_by_type.png
✅ Saved: baseload_vs_peak.png
✅ Saved: correlation_heatmap.png
✅ Saved: summary_stats.json

✅ All plots saved to wasbs://results@dnvenergystorage.blob.core.windows.net/plots/


8. MODEL TRAINING
 
XGBoost + scikit-learn.
- K-Means: 3 clusters, selected with elbow method. Shilloute score of 0.64. Three groups: 1306 standard operators, 141 efficient buildings (low baseload and weekend ratios) and 87 high baseload operators, with really high baseloads and weekend ratios. In this category we may find big data centers or hospials, but also regular buildings that are overspending -> Priority targets for demand side programs

- Isolation Forest: assumed that 15% of the buildings are anomalies, and used it in combination with K-Means to find priority buildings

- XGBoost: Used XGboost to predict weekend consumptions using the current features. The compared the prediction with the actual value to estimate how much a building can be improved on the weekends, by flaggine buildings which weekend ratio is 10% higher than predicted as underperforming. (41 underperformers)
MAE: 0.0361, R²: 0.9236
The train split is suboptimal because I was amximizing the model utility now, and trained on the entire dataset. In the 2 first methods its fine because we are not predicting, but explaining the current datasets. The SGboost I would use a time-series cross validation to see the real predictive power for time-series data.

The top priority buildings are later estimated as a weighted combination of the 3 models.

In [0]:
print("=" * 60)
print("STAGE 8: MODEL TRAINING")
print("=" * 60)

import sklearn
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import silhouette_score, mean_absolute_error, r2_score
import xgboost as xgb

# -----------------------------------------------------------------------------
# MODEL 1: K-Means Clustering
# -----------------------------------------------------------------------------
import sys
import contextlib
import os

@contextlib.contextmanager
def suppress_stderr():
    with open(os.devnull, "w") as f:
        old_stderr = sys.stderr
        sys.stderr = f
        try:
            yield
        finally:
            sys.stderr = old_stderr


print("\n🎯 MODEL 1: K-MEANS CLUSTERING")

cluster_features = ['baseload', 'avg_peak_ratio', 'avg_consumption', 
                   'weekend_ratio', 'night_ratio', 'consumption_cv']
X_cluster = df_bldg[cluster_features].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Test k=3,4,5,6
for k in range(3, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    with suppress_stderr():
        labels = km.fit_predict(X_scaled)
    print(f"   k={k}: Silhouette={silhouette_score(X_scaled, labels):.3f}")

# Use k=3 for interpretability
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_bldg['cluster'] = kmeans.fit_predict(X_scaled)
sil_score = silhouette_score(X_scaled, df_bldg['cluster'])

print(f"✅ Selected k=3, Silhouette: {sil_score:.3f}")
print(f"   Clusters: {df_bldg['cluster'].value_counts().sort_index().to_dict()}")

# -----------------------------------------------------------------------------
# MODEL 2: Isolation Forest
# -----------------------------------------------------------------------------
print("\n🔍 MODEL 2: ISOLATION FOREST")

anomaly_features = ['baseload', 'avg_peak_ratio', 'avg_consumption',
                   'weekend_ratio', 'night_ratio', 'consumption_cv',
                   'temp_sensitivity', 'avg_volatility']
X_anomaly = df_bldg[anomaly_features].fillna(0)

iso = IsolationForest(contamination=0.15, random_state=42, n_estimators=100)
df_bldg['is_anomaly'] = (iso.fit_predict(X_anomaly) == -1).astype(int)
n_anomalies = df_bldg['is_anomaly'].sum()

print(f"✅ Anomalies: {n_anomalies} ({n_anomalies/len(df_bldg)*100:.1f}%)")

# -----------------------------------------------------------------------------
# MODEL 3: XGBoost
# -----------------------------------------------------------------------------
print("\n📈 MODEL 3: XGBOOST")

df_model = df_bldg.dropna(subset=['weekend_ratio']).copy()
le = LabelEncoder()
df_model['building_type_encoded'] = le.fit_transform(df_model['building_type'].fillna('unknown'))

xgb_features = ['avg_consumption', 'baseload', 'avg_peak_ratio', 'night_ratio', 
               'consumption_cv', 'avg_volatility', 'temp_sensitivity', 'building_type_encoded']
X_train = df_model[xgb_features].fillna(0)
y_train = df_model['weekend_ratio']

xgb_model = xgb.XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.05, 
                             random_state=42, subsample=0.8, colsample_bytree=0.8)
xgb_model.fit(X_train, y_train)

predictions = xgb_model.predict(X_train)
mae = mean_absolute_error(y_train, predictions)
r2 = r2_score(y_train, predictions)

df_model['predicted_weekend_ratio'] = predictions
df_model['weekend_gap'] = df_model['weekend_ratio'] - predictions
df_model['underperformer'] = (df_model['weekend_gap'] > 0.10).astype(int)

print(f"✅ MAE: {mae:.4f}, R²: {r2:.4f}")
print(f"   Underperformers: {df_model['underperformer'].sum()}")

# Merge predictions back
df_bldg = df_bldg.merge(
    df_model[['building_id', 'predicted_weekend_ratio', 'weekend_gap', 'underperformer']],
    on='building_id', how='left'
)

print("\n✅ Stage 8 Complete: All models trained")

STAGE 8: MODEL TRAINING


<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead
<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead



🎯 MODEL 1: K-MEANS CLUSTERING


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

   k=3: Silhouette=0.637


...

✅ Selected k=3, Silhouette: 0.637
   Clusters: {0: 1306, 1: 141, 2: 87}

🔍 MODEL 2: ISOLATION FOREST


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Anomalies: 230 (15.0%)

📈 MODEL 3: XGBOOST


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

✅ MAE: 0.0361, R²: 0.9236
   Underperformers: 41

✅ Stage 8 Complete: All models trained


9. EXPERIMENT TRACKING


Use MLFlow to log the models and their performance metrics.

In [0]:
print("=" * 60)
print("STAGE 9: EXPERIMENT TRACKING (MLflow)")
print("=" * 60)

import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models import infer_signature

mlflow.set_experiment("/building-energy-ml-v2")

# Log K-Means
print("\n📝 Logging K-Means...")
with mlflow.start_run(run_name="kmeans_clustering"):
    mlflow.log_param("n_clusters", 3)
    mlflow.log_param("features", ", ".join(cluster_features))
    mlflow.log_metric("silhouette_score", sil_score)
    mlflow.sklearn.log_model(kmeans, "kmeans_model")
print("   ✅ K-Means logged")

# Log Isolation Forest
print("\n📝 Logging Isolation Forest...")
with mlflow.start_run(run_name="isolation_forest"):
    mlflow.log_param("contamination", 0.15)
    mlflow.log_param("features", ", ".join(anomaly_features))
    mlflow.log_metric("n_anomalies", n_anomalies)
    mlflow.sklearn.log_model(iso, "isolation_forest_model")
print("   ✅ Isolation Forest logged")

# Log XGBoost
print("\n📝 Logging XGBoost...")
with mlflow.start_run(run_name="xgboost_weekend") as run:
    mlflow.log_params({"n_estimators": 150, "max_depth": 5, "learning_rate": 0.05})
    mlflow.log_param("features", ", ".join(xgb_features))
    mlflow.log_metrics({"mae": mae, "r2_score": r2})
    
    signature = infer_signature(X_train, predictions)
    mlflow.xgboost.log_model(xgb_model, "xgboost_model", signature=signature)
    
    xgb_run_id = run.info.run_id
print("   ✅ XGBoost logged")

print("\n✅ Stage 9 Complete: All experiments tracked")

STAGE 9: EXPERIMENT TRACKING (MLflow)

📝 Logging K-Means...


2026/01/10 07:36:33 WARNING mlflow.models.model: Model logged without a signature. Signatures will be required for upcoming model registry features as they validate model inputs and denote the expected schema of model outputs. Please visit https://www.mlflow.org/docs/2.9.2/models.html#set-signature-on-logged-model for instructions on setting a model signature on your logged model.
/databricks/python/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

   ✅ K-Means logged

📝 Logging Isolation Forest...


2026/01/10 07:36:38 WARNING mlflow.models.model: Model logged without a signature. Signatures will be required for upcoming model registry features as they validate model inputs and denote the expected schema of model outputs. Please visit https://www.mlflow.org/docs/2.9.2/models.html#set-signature-on-logged-model for instructions on setting a model signature on your logged model.


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

   ✅ Isolation Forest logged

📝 Logging XGBoost...


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

   ✅ XGBoost logged

✅ Stage 9 Complete: All experiments tracked


10. MODEL EVALUATION


Print evaluation metrics of the different models and its explainabiilty.

In [0]:
print("=" * 60)
print("STAGE 10: MODEL EVALUATION + SHAP")
print("=" * 60)

import shap
import json
import pandas as pd

# -----------------------------------------------------------------------------
# 10.1: Metrics Summary
# -----------------------------------------------------------------------------
print("\n📊 Model Metrics Summary:")
print(f"   K-Means:         Silhouette = {sil_score:.3f}")
print(f"   Isolation Forest: Anomalies = {n_anomalies} ({n_anomalies/len(df_bldg)*100:.1f}%)")
print(f"   XGBoost:          MAE = {mae:.4f}, R² = {r2:.4f}")

# -----------------------------------------------------------------------------
# 10.2: Cluster Evaluation
# -----------------------------------------------------------------------------
print("\n📊 Cluster Profiles:")
for i in range(3):
    cluster = df_bldg[df_bldg['cluster'] == i]
    print(f"   Cluster {i}: {len(cluster)} buildings, avg_baseload={cluster['baseload'].mean():.1f}")

# -----------------------------------------------------------------------------
# 10.3: SHAP Explainability
# -----------------------------------------------------------------------------
print("\n🔬 Computing SHAP values...")

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_train)

# Top features by importance
shap_importance = dict(zip(xgb_features, abs(shap_values).mean(axis=0)))
print("\n   Top SHAP features:")
for feat, imp in sorted(shap_importance.items(), key=lambda x: -x[1])[:5]:
    print(f"   • {feat}: {imp:.4f}")

# Store SHAP as JSON for each building
shap_records = []
for idx, row in df_model.iterrows():
    shap_dict = {feat: float(shap_values[df_model.index.get_loc(idx), i]) 
                 for i, feat in enumerate(xgb_features)}
    shap_records.append({'building_id': row['building_id'], 'shap_json': json.dumps(shap_dict)})

df_shap = pd.DataFrame(shap_records)
df_bldg = df_bldg.merge(df_shap, on='building_id', how='left')

print("\n✅ Stage 10 Complete: Evaluation and SHAP computed")

STAGE 10: MODEL EVALUATION + SHAP


/databricks/python_shell/lib/dbruntime/PostImportHook.py:295: DeprecationWarning: Deprecated since Python 3.4 and slated for removal in Python 3.12; use importlib.util.find_spec() instead
  loader = importlib.find_loader(fullname, path)



📊 Model Metrics Summary:
   K-Means:         Silhouette = 0.637
   Isolation Forest: Anomalies = 230 (15.0%)
   XGBoost:          MAE = 0.0361, R² = 0.9236

📊 Cluster Profiles:
   Cluster 0: 1306 buildings, avg_baseload=76.2
   Cluster 1: 141 buildings, avg_baseload=6.7
   Cluster 2: 87 buildings, avg_baseload=758.0

🔬 Computing SHAP values...

   Top SHAP features:
   • night_ratio: 0.0699
   • building_type_encoded: 0.0262
   • consumption_cv: 0.0242
   • avg_volatility: 0.0231
   • temp_sensitivity: 0.0150

✅ Stage 10 Complete: Evaluation and SHAP computed


11. MODEL REGISTRY

Register XGBoost on MLFlow Registry for inference.

In [0]:
print("=" * 60)
print("STAGE 11: MODEL REGISTRY")
print("=" * 60)

# Register XGBoost (the model we'll serve for inference)
print("\n📦 Registering XGBoost to MLflow Registry...")

model_name = "building-weekend-efficiency-predictor"
mlflow.register_model(f"runs:/{xgb_run_id}/xgboost_model", model_name)

print(f"   ✅ Model registered: {model_name}")
print(f"   Run ID: {xgb_run_id}")

# Note: K-Means and Isolation Forest not registered because their outputs
# (cluster assignments, anomaly flags) are already stored in the scored data.
# Register them if you need to re-cluster/re-detect on new buildings.

print("\n✅ Stage 11 Complete: Model registered")

STAGE 11: MODEL REGISTRY

📦 Registering XGBoost to MLflow Registry...


Registered model 'building-weekend-efficiency-predictor' already exists. Creating a new version of this model...
The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.


Uploading artifacts:   0%|          | 0/5 [00:00<?, ?it/s]

   ✅ Model registered: building-weekend-efficiency-predictor
   Run ID: 5bff8f7a858f401696ad5f5617bb0fc1

✅ Stage 11 Complete: Model registered


Created version '7' of model 'dnv_energy_demo.default.building-weekend-efficiency-predictor'.


12. DEPLOY

Streamlit dashboard that consumes a FastAPI backend, which serves pre-computed predictions from a parquet file. Everything runs locally.


**Production Deployment**

For production, you would containerize both the FastAPI and Streamlit into Docker containers. I would just package the Python code and dependencies into an image. Then, push this image to Azure Container Registry (ACR), which is Azure storage for containers. Finally deploy in Azure COntainer Instance (ACI), which would expose the container in a public URL.

This would be better for production because the databricks pipleine would run on a schedule (daily, weekly, etc), and redo predictions and scores, saving the new reuslts in Azure Storage. The API, when restarted, would load the new data, or have and endpoint that refresh the data.
 
**Scaling Options**

ACI is ideal for simple, low-traffic APIs serving pre-computed results - it's cost-effective and easy to manage. If needed high traffic handling with auto-scaling and multiple services, Azure Kubernetes Service (AKS) would be the choice, though it adds complexity and cost.


**Docker vs Kubernetes**

Docker is the container technology itself - it packages your application and dependencies into an isolated, portable unit. Think of it as a lightweight virtual machine that runs the same everywhere. ACI runs Docker containers directly, which is sufficient for simple deployments. Kubernetes (and AKS) is a container orchestration platform - it manages multiple Docker containers, automatically restarts failed ones, scales up when traffic increases, load balances across instances, and handles rolling updates without downtime. You'd use Kubernetes when you have multiple microservices that need to communicate, require high availability, or expect variable traffic patterns. For a single API serving batch predictions to a dashboard, Docker on ACI is simpler and appropriate. Kubernetes becomes valuable when the system grows - multiple models, multiple APIs, hundreds of concurrent users.


**Real-Time vs Batch**

Azure ML Endpoints is a different approach entirely. Instead of serving pre-computed predictions, you deploy the actual XGBoost model and run inference at request time - a new building comes in, you call the endpoint with its features, and it returns a prediction live. This makes sense when you have new buildings arriving frequently and can't wait for the next batch run. For our use case with a relatively static building portfolio scored periodically, batch inference with ACI is simpler and more cost-effective.


In [0]:
print("=" * 60)
print("STAGE 12: BATCH SCORING & EXPORT")
print("=" * 60)
from datetime import datetime
import numpy as np

# -----------------------------------------------------------------------------
# 12.1: Identify Inefficient Cluster
# -----------------------------------------------------------------------------
inefficient_cluster = df_bldg.groupby('cluster')['baseload'].mean().idxmax()
print(f"\n🔍 Inefficient cluster identified: {inefficient_cluster}")

# -----------------------------------------------------------------------------
# 12.2: Generate Recommendations
# -----------------------------------------------------------------------------
print("💡 Generating recommendations...")

def get_recommendation(row):
    recs = []
    if row['cluster'] == inefficient_cluster:
        recs.append("PRIORITY: High baseload cluster - immediate audit")
    if row['is_anomaly'] == 1 and row['avg_consumption'] > min_consumption:
        recs.append("ANOMALY: Unusual pattern detected - check equipment/controls")
    if row.get('underperformer') == 1:
        recs.append(f"EFFICIENCY: Weekend gap {row.get('weekend_gap', 0):.0%} above expected")
    return "; ".join(recs) if recs else "STANDARD: Normal operation"

# -----------------------------------------------------------------------------
# 12.3: Priority Scoring (4 components, 25 pts each, max 100)
# -----------------------------------------------------------------------------
print("🎯 Calculating priority scores...")

# Minimum consumption threshold - bottom 25% excluded from anomaly scoring
min_consumption = df_bldg['avg_consumption'].quantile(0.25)
print(f"   Min consumption threshold: {min_consumption:.1f} kWh")

# Component 1: In inefficient cluster? (25 pts)
in_bad_cluster = (df_bldg['cluster'] == inefficient_cluster).astype(int) * 30

# Component 2: Has efficiency gap from XGBoost? (25 pts)
gap_threshold = 0.05  # 10% worse than predicted
has_efficiency_gap = (df_bldg['weekend_gap'].fillna(0) > gap_threshold).astype(int) * 25

# Component 3: Consumption percentile - higher consumption = higher priority (0-25 pts)
consumption_score = df_bldg['avg_consumption'].rank(pct=True) * 10

# Component 4: Anomaly only if consumption is meaningful (25 pts)
meaningful_anomaly = (
    (df_bldg['is_anomaly'].fillna(0) == 1) & 
    (df_bldg['avg_consumption'] > min_consumption)
).astype(int) * 30

# Combine scores
df_bldg['priority_score'] = (
    in_bad_cluster + 
    has_efficiency_gap + 
    consumption_score + 
    meaningful_anomaly
).round(2)

# Generate recommendations (after min_consumption is defined)
df_bldg['recommendation'] = df_bldg.apply(get_recommendation, axis=1)

# Rank buildings
df_bldg['priority_rank'] = df_bldg['priority_score'].rank(ascending=False, method='dense').astype(int)
df_bldg['scored_at'] = datetime.now().isoformat()

# Print score breakdown
print(f"\n   Score components (25 pts each):")
print(f"   - In inefficient cluster: {in_bad_cluster.sum() // 25} buildings")
print(f"   - Has efficiency gap (>{gap_threshold:.0%}): {(has_efficiency_gap > 0).sum()} buildings")
print(f"   - Meaningful anomalies: {meaningful_anomaly.sum() // 25} buildings")
print(f"   - Consumption: scaled 0-25 by percentile")

# -----------------------------------------------------------------------------
# 12.4: Export for API
# -----------------------------------------------------------------------------
print("\n📦 Exporting predictions...")
api_columns = ['building_id', 'building_type', 'avg_consumption', 'baseload', 
               'weekend_ratio', 'night_ratio', 'cluster', 'is_anomaly',
               'predicted_weekend_ratio', 'weekend_gap', 'underperformer',
               'shap_json', 'recommendation', 'priority_rank', 'priority_score', 'scored_at']
df_api = df_bldg[[c for c in api_columns if c in df_bldg.columns]].sort_values('priority_rank')

# Save locally
df_api.to_parquet("/tmp/predictions.parquet", index=False)
df_api.to_csv("/tmp/predictions.csv", index=False)

# Copy to Azure Blob Storage
dbutils.fs.cp("file:/tmp/predictions.parquet", f"{SAVE_PATH}/predictions.parquet")
dbutils.fs.cp("file:/tmp/predictions.csv", f"{SAVE_PATH}/predictions.csv")

print(f"✅ Exported {len(df_api)} buildings")
print(f"   High priority (score >= 75): {len(df_api[df_api['priority_score'] >= 75])}")
print(f"   Medium priority (50-75): {len(df_api[(df_api['priority_score'] >= 50) & (df_api['priority_score'] < 75)])}")
print(f"   Saved to: {SAVE_PATH}/predictions.parquet")
print("\n✅ Stage 12 Complete: Batch scoring and export done")

STAGE 12: BATCH SCORING & EXPORT

🔍 Inefficient cluster identified: 2
💡 Generating recommendations...
🎯 Calculating priority scores...
   Min consumption threshold: 20.2 kWh

   Score components (25 pts each):
   - In inefficient cluster: 104 buildings
   - Has efficiency gap (>5%): 191 buildings
   - Meaningful anomalies: 148 buildings
   - Consumption: scaled 0-25 by percentile

📦 Exporting predictions...
✅ Exported 1534 buildings
   High priority (score >= 75): 5
   Medium priority (50-75): 69
   Saved to: wasbs://results@dnvenergystorage.blob.core.windows.net/predictions.parquet

✅ Stage 12 Complete: Batch scoring and export done


SUMMARY

In [0]:
print("=" * 60)
print("✅ PIPELINE COMPLETE")
print("=" * 60)

print(f"""
📊 DATASET:
   • Buildings: {len(df_bldg)}
   • Features: {len(df_bldg.columns)}

🎯 MODEL 1 - K-MEANS:
   • Clusters: 3
   • Silhouette: {sil_score:.3f}

🔍 MODEL 2 - ISOLATION FOREST:
   • Anomalies: {n_anomalies} ({n_anomalies/len(df_bldg)*100:.1f}%)

📈 MODEL 3 - XGBOOST:
   • MAE: {mae:.4f}, R²: {r2:.4f}
   • Underperformers: {df_model['underperformer'].sum()}

💰 BUSINESS IMPACT:
   • Addressable buildings: ~{len(df_bldg[df_bldg['is_anomaly']==1]) + int(df_model['underperformer'].sum())} ({(len(df_bldg[df_bldg['is_anomaly']==1]) + df_model['underperformer'].sum())/len(df_bldg)*100:.0f}%)

📦 OUTPUTS:
   • predictions.parquet - All buildings with scores + SHAP
   • Model registered to MLflow Registry
""")

✅ PIPELINE COMPLETE

📊 DATASET:
   • Buildings: 1534
   • Features: 29

🎯 MODEL 1 - K-MEANS:
   • Clusters: 3
   • Silhouette: 0.637

🔍 MODEL 2 - ISOLATION FOREST:
   • Anomalies: 230 (15.0%)

📈 MODEL 3 - XGBOOST:
   • MAE: 0.0361, R²: 0.9236
   • Underperformers: 41

💰 BUSINESS IMPACT:
   • Addressable buildings: ~271 (18%)

📦 OUTPUTS:
   • predictions.parquet - All buildings with scores + SHAP
   • Model registered to MLflow Registry

